# SMS Spam Classification - Model Training and Evaluation
This notebook trains, evaluates, and selects the best spam classification model.

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score
from sklearn.metrics import average_precision_score
from sklearn.model_selection import GridSearchCV
import warnings
warnings.filterwarnings('ignore')

# MLflow imports
import mlflow
import mlflow.sklearn
import os
from datetime import datetime

## 1. Load Data

In [2]:
# Load the prepared datasets
train_df = pd.read_csv('train.csv')
val_df = pd.read_csv('validation.csv')
test_df = pd.read_csv('test.csv')

print(f"Train set: {len(train_df)} samples")
print(f"Validation set: {len(val_df)} samples")
print(f"Test set: {len(test_df)} samples")

Train set: 4135 samples
Validation set: 517 samples
Test set: 517 samples


In [3]:
# Prepare features and labels
X_train = train_df['message']
y_train = train_df['label_binary']

X_val = val_df['message']
y_val = val_df['label_binary']

X_test = test_df['message']
y_test = test_df['label_binary']

## 2. Feature Extraction

In [4]:
# Create TF-IDF vectorizer
vectorizer = TfidfVectorizer(max_features=3000, stop_words='english', ngram_range=(1, 2))

# Fit on training data and transform all sets
X_train_tfidf = vectorizer.fit_transform(X_train)
X_val_tfidf = vectorizer.transform(X_val)
X_test_tfidf = vectorizer.transform(X_test)

print(f"Feature matrix shape: {X_train_tfidf.shape}")
print(f"Number of features: {X_train_tfidf.shape[1]}")

Feature matrix shape: (4135, 3000)
Number of features: 3000


## 2a. MLflow Setup for Experiment Tracking

In [ ]:
# Initialize MLflow
mlflow.set_tracking_uri(os.getcwd())
experiment_name = "SMS-Spam-Classification"

# Create or get experiment
try:
    experiment_id = mlflow.create_experiment(experiment_name)
except:
    experiment = mlflow.get_experiment_by_name(experiment_name)
    experiment_id = experiment.experiment_id

mlflow.set_experiment(experiment_name)
print(f"✓ MLflow experiment '{experiment_name}' initialized")
print(f"  Experiment ID: {experiment_id}")
print(f"  Tracking URI: {mlflow.get_tracking_uri()}")

## 3. Define Helper Functions

In [5]:
def fit_model(model, X_train, y_train):
    """
    Fit a model on training data.
    
    Parameters:
    -----------
    model : sklearn classifier
        Model to train
    X_train : array-like
        Training features
    y_train : array-like
        Training labels
        
    Returns:
    --------
    fitted model
    """
    model.fit(X_train, y_train)
    return model

In [ ]:
def train_and_log_model(model_name, model, X_train, y_train, X_val, y_val, X_test, y_test):
    """
    Train model and log metrics to MLflow.
    
    Parameters:
    -----------
    model_name : str
        Name of the model for MLflow
    model : sklearn classifier
        Model to train
    X_train, y_train : features and labels for training
    X_val, y_val : features and labels for validation
    X_test, y_test : features and labels for test
        
    Returns:
    --------
    tuple
        (fitted_model, run_id, metrics)
    """
    with mlflow.start_run(run_name=model_name):
        # Fit model
        print(f"\n{'='*60}")
        print(f"Training {model_name}...")
        print(f"{'='*60}")
        
        fitted_model = fit_model(model, X_train, y_train)
        
        # Score on train
        train_scores = score_model(fitted_model, X_train, y_train)
        train_metrics = evaluate_predictions(
            y_train, 
            train_scores['predictions'], 
            train_scores['probabilities'],
            f"{model_name} - Training Set"
        )
        
        # Score on validation
        val_scores = score_model(fitted_model, X_val, y_val)
        val_metrics = evaluate_predictions(
            y_val, 
            val_scores['predictions'], 
            val_scores['probabilities'],
            f"{model_name} - Validation Set"
        )
        
        # Score on test
        test_scores = score_model(fitted_model, X_test, y_test)
        test_metrics = evaluate_predictions(
            y_test, 
            test_scores['predictions'], 
            test_scores['probabilities'],
            f"{model_name} - Test Set"
        )
        
        # Calculate AUCPR (Average Precision Score) for all sets
        train_aucpr = average_precision_score(y_train, train_scores['probabilities']) if train_scores['probabilities'] is not None else 0
        val_aucpr = average_precision_score(y_val, val_scores['probabilities']) if val_scores['probabilities'] is not None else 0
        test_aucpr = average_precision_score(y_test, test_scores['probabilities']) if test_scores['probabilities'] is not None else 0
        
        # Log training parameters
        mlflow.log_param("model_type", type(model).__name__)
        
        # Log training metrics
        mlflow.log_metric("train_accuracy", train_metrics['accuracy'])
        mlflow.log_metric("train_precision", train_metrics['precision'])
        mlflow.log_metric("train_recall", train_metrics['recall'])
        mlflow.log_metric("train_f1_score", train_metrics['f1_score'])
        mlflow.log_metric("train_auc", train_metrics.get('auc', 0))
        mlflow.log_metric("train_aucpr", train_aucpr)
        
        # Log validation metrics
        mlflow.log_metric("val_accuracy", val_metrics['accuracy'])
        mlflow.log_metric("val_precision", val_metrics['precision'])
        mlflow.log_metric("val_recall", val_metrics['recall'])
        mlflow.log_metric("val_f1_score", val_metrics['f1_score'])
        mlflow.log_metric("val_auc", val_metrics.get('auc', 0))
        mlflow.log_metric("val_aucpr", val_aucpr)
        
        # Log test metrics
        mlflow.log_metric("test_accuracy", test_metrics['accuracy'])
        mlflow.log_metric("test_precision", test_metrics['precision'])
        mlflow.log_metric("test_recall", test_metrics['recall'])
        mlflow.log_metric("test_f1_score", test_metrics['f1_score'])
        mlflow.log_metric("test_auc", test_metrics.get('auc', 0))
        mlflow.log_metric("test_aucpr", test_aucpr)
        
        # Log model with MLflow
        mlflow.sklearn.log_model(fitted_model, f"{model_name}_model")
        
        # Get current run
        run_id = mlflow.active_run().info.run_id
        mlflow.end_run()
        
        print(f"\n✓ {model_name} logged to MLflow (Run ID: {run_id})")
        print(f"  Val AUCPR: {val_aucpr:.4f}")
        
        return fitted_model, run_id, {
            'train': train_metrics,
            'val': val_metrics,
            'test': test_metrics,
            'train_aucpr': train_aucpr,
            'val_aucpr': val_aucpr,
            'test_aucpr': test_aucpr
        }

In [6]:
def score_model(model, X, y):
    """
    Score a model on given data.
    
    Parameters:
    -----------
    model : sklearn classifier
        Fitted model
    X : array-like
        Features
    y : array-like
        True labels
        
    Returns:
    --------
    dict
        Dictionary containing predictions and probabilities
    """
    predictions = model.predict(X)
    
    # Get probability predictions if available
    if hasattr(model, 'predict_proba'):
        probabilities = model.predict_proba(X)[:, 1]
    elif hasattr(model, 'decision_function'):
        probabilities = model.decision_function(X)
    else:
        probabilities = None
    
    return {
        'predictions': predictions,
        'probabilities': probabilities
    }

In [7]:
def evaluate_predictions(y_true, y_pred, y_proba=None, dataset_name="Dataset"):
    """
    Evaluate model predictions and print metrics.
    
    Parameters:
    -----------
    y_true : array-like
        True labels
    y_pred : array-like
        Predicted labels
    y_proba : array-like, optional
        Predicted probabilities
    dataset_name : str
        Name of the dataset for display
        
    Returns:
    --------
    dict
        Dictionary containing all metrics
    """
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    
    metrics = {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1_score': f1
    }
    
    # Calculate AUC if probabilities are available
    if y_proba is not None:
        auc = roc_auc_score(y_true, y_proba)
        metrics['auc'] = auc
    
    # Print results
    print(f"\n{'='*50}")
    print(f"Performance on {dataset_name}")
    print(f"{'='*50}")
    print(f"Accuracy:  {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"F1-Score:  {f1:.4f}")
    if y_proba is not None:
        print(f"AUC:       {auc:.4f}")
    
    # Confusion matrix
    cm = confusion_matrix(y_true, y_pred)
    print(f"\nConfusion Matrix:")
    print(f"                 Predicted")
    print(f"              Ham    Spam")
    print(f"Actual Ham    {cm[0,0]:<6} {cm[0,1]:<6}")
    print(f"       Spam   {cm[1,0]:<6} {cm[1,1]:<6}")
    
    return metrics

In [8]:
def validate_model(model, X_train, y_train, X_val, y_val):
    """
    Fit model on training data and evaluate on both train and validation sets.
    
    Parameters:
    -----------
    model : sklearn classifier
        Model to train and validate
    X_train : array-like
        Training features
    y_train : array-like
        Training labels
    X_val : array-like
        Validation features
    y_val : array-like
        Validation labels
        
    Returns:
    --------
    tuple
        (fitted_model, train_metrics, val_metrics)
    """
    # Fit model
    fitted_model = fit_model(model, X_train, y_train)
    
    # Score on train
    train_scores = score_model(fitted_model, X_train, y_train)
    train_metrics = evaluate_predictions(
        y_train, 
        train_scores['predictions'], 
        train_scores['probabilities'],
        "Training Set"
    )
    
    # Score on validation
    val_scores = score_model(fitted_model, X_val, y_val)
    val_metrics = evaluate_predictions(
        y_val, 
        val_scores['predictions'], 
        val_scores['probabilities'],
        "Validation Set"
    )
    
    return fitted_model, train_metrics, val_metrics

## 4. Train and Evaluate Benchmark Models

### 4.1 Naive Bayes Classifier

In [ ]:
# Create and validate Naive Bayes model with MLflow tracking
nb_model = MultinomialNB()
nb_fitted, nb_run_id, nb_all_metrics = train_and_log_model(
    "Naive Bayes", nb_model, 
    X_train_tfidf, y_train, 
    X_val_tfidf, y_val,
    X_test_tfidf, y_test
)
nb_train_metrics = nb_all_metrics['train']
nb_val_metrics = nb_all_metrics['val']
nb_test_metrics = nb_all_metrics['test']


Performance on Training Set
Accuracy:  0.9838
Precision: 1.0000
Recall:    0.8716
F1-Score:  0.9314
AUC:       0.9948

Confusion Matrix:
                 Predicted
              Ham    Spam
Actual Ham    3613   0     
       Spam   67     455   

Performance on Validation Set
Accuracy:  0.9710
Precision: 1.0000
Recall:    0.7727
F1-Score:  0.8718
AUC:       0.9861

Confusion Matrix:
                 Predicted
              Ham    Spam
Actual Ham    451    0     
       Spam   15     51    


### 4.2 Logistic Regression

In [ ]:
# Create and validate Logistic Regression model with MLflow tracking
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_fitted, lr_run_id, lr_all_metrics = train_and_log_model(
    "Logistic Regression", lr_model,
    X_train_tfidf, y_train,
    X_val_tfidf, y_val,
    X_test_tfidf, y_test
)
lr_train_metrics = lr_all_metrics['train']
lr_val_metrics = lr_all_metrics['val']
lr_test_metrics = lr_all_metrics['test']


Performance on Training Set
Accuracy:  0.9739
Precision: 0.9905
Recall:    0.8008
F1-Score:  0.8856
AUC:       0.9978

Confusion Matrix:
                 Predicted
              Ham    Spam
Actual Ham    3609   4     
       Spam   104    418   

Performance on Validation Set
Accuracy:  0.9574
Precision: 1.0000
Recall:    0.6667
F1-Score:  0.8000
AUC:       0.9889

Confusion Matrix:
                 Predicted
              Ham    Spam
Actual Ham    451    0     
       Spam   22     44    


### 4.3 Support Vector Machine (SVM)

In [ ]:
# Create and validate SVM model with MLflow tracking
svm_model = LinearSVC(random_state=42, max_iter=2000)
svm_fitted, svm_run_id, svm_all_metrics = train_and_log_model(
    "SVM", svm_model,
    X_train_tfidf, y_train,
    X_val_tfidf, y_val,
    X_test_tfidf, y_test
)
svm_train_metrics = svm_all_metrics['train']
svm_val_metrics = svm_all_metrics['val']
svm_test_metrics = svm_all_metrics['test']


Performance on Training Set
Accuracy:  0.9983
Precision: 0.9981
Recall:    0.9885
F1-Score:  0.9933
AUC:       0.9997

Confusion Matrix:
                 Predicted
              Ham    Spam
Actual Ham    3612   1     
       Spam   6      516   

Performance on Validation Set
Accuracy:  0.9787
Precision: 0.9661
Recall:    0.8636
F1-Score:  0.9120
AUC:       0.9917

Confusion Matrix:
                 Predicted
              Ham    Spam
Actual Ham    449    2     
       Spam   9      57    


## 5. Compare Validation Performance

In [12]:
# Create comparison table
comparison_df = pd.DataFrame({
    'Model': ['Naive Bayes', 'Logistic Regression', 'SVM'],
    'Train F1': [nb_train_metrics['f1_score'], lr_train_metrics['f1_score'], svm_train_metrics['f1_score']],
    'Val F1': [nb_val_metrics['f1_score'], lr_val_metrics['f1_score'], svm_val_metrics['f1_score']],
    'Train Acc': [nb_train_metrics['accuracy'], lr_train_metrics['accuracy'], svm_train_metrics['accuracy']],
    'Val Acc': [nb_val_metrics['accuracy'], lr_val_metrics['accuracy'], svm_val_metrics['accuracy']],
    'Val Precision': [nb_val_metrics['precision'], lr_val_metrics['precision'], svm_val_metrics['precision']],
    'Val Recall': [nb_val_metrics['recall'], lr_val_metrics['recall'], svm_val_metrics['recall']]
})

print("\n" + "="*80)
print("MODEL COMPARISON ON VALIDATION SET")
print("="*80)
print(comparison_df.to_string(index=False))
print("="*80)


MODEL COMPARISON ON VALIDATION SET
              Model  Train F1   Val F1  Train Acc  Val Acc  Val Precision  Val Recall
        Naive Bayes  0.931423 0.871795   0.983797 0.970986       1.000000    0.772727
Logistic Regression  0.885593 0.800000   0.973881 0.957447       1.000000    0.666667
                SVM  0.993263 0.912000   0.998307 0.978723       0.966102    0.863636


## 6. Hyperparameter Tuning (Best Model)

In [13]:
# Fine-tune SVM (LinearSVC) hyperparameters
print("\nFine-tuning SVM hyperparameters...")

param_grid = {
    'C': [0.1, 1.0, 10.0],
    'loss': ['hinge', 'squared_hinge'],
    'max_iter': [2000, 3000]
}

grid_search = GridSearchCV(
    LinearSVC(random_state=42),
    param_grid,
    cv=3,
    scoring='f1',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train_tfidf, y_train)

print(f"\nBest parameters: {grid_search.best_params_}")
print(f"Best cross-validation F1 score: {grid_search.best_score_:.4f}")

# Get the best model
best_lr_model = grid_search.best_estimator_


Fine-tuning SVM hyperparameters...
Fitting 3 folds for each of 12 candidates, totalling 36 fits

Best parameters: {'C': 1.0, 'loss': 'squared_hinge', 'max_iter': 2000}
Best cross-validation F1 score: 0.9155


In [14]:
# Evaluate tuned model on validation set
val_scores_tuned = score_model(best_lr_model, X_val_tfidf, y_val)
tuned_val_metrics = evaluate_predictions(
    y_val,
    val_scores_tuned['predictions'],
    val_scores_tuned['probabilities'],
    "Validation Set (Tuned LR)"
)


Performance on Validation Set (Tuned LR)
Accuracy:  0.9787
Precision: 0.9661
Recall:    0.8636
F1-Score:  0.9120
AUC:       0.9917

Confusion Matrix:
                 Predicted
              Ham    Spam
Actual Ham    449    2     
       Spam   9      57    


## 7. Final Evaluation on Test Set

In [15]:
# Evaluate all three benchmark models on test set
print("\n" + "="*80)
print("FINAL TEST SET EVALUATION")
print("="*80)

# Store results
test_results = []

# 1. Naive Bayes
nb_test_scores = score_model(nb_fitted, X_test_tfidf, y_test)
nb_test_metrics = evaluate_predictions(
    y_test,
    nb_test_scores['predictions'],
    nb_test_scores['probabilities'],
    "Test Set - Naive Bayes"
)
test_results.append(('Naive Bayes', nb_test_metrics))

# 2. Logistic Regression (original)
lr_test_scores = score_model(lr_fitted, X_test_tfidf, y_test)
lr_test_metrics = evaluate_predictions(
    y_test,
    lr_test_scores['predictions'],
    lr_test_scores['probabilities'],
    "Test Set - Logistic Regression"
)
test_results.append(('Logistic Regression', lr_test_metrics))

# 3. SVM
svm_test_scores = score_model(svm_fitted, X_test_tfidf, y_test)
svm_test_metrics = evaluate_predictions(
    y_test,
    svm_test_scores['predictions'],
    svm_test_scores['probabilities'],
    "Test Set - SVM"
)
test_results.append(('SVM', svm_test_metrics))

# 4. Tuned SVM
tuned_test_scores = score_model(best_lr_model, X_test_tfidf, y_test)
tuned_test_metrics = evaluate_predictions(
    y_test,
    tuned_test_scores['predictions'],
    tuned_test_scores['probabilities'],
    "Test Set - Tuned SVM"
)
test_results.append(('Tuned SVM', tuned_test_metrics))


FINAL TEST SET EVALUATION

Performance on Test Set - Naive Bayes
Accuracy:  0.9787
Precision: 1.0000
Recall:    0.8308
F1-Score:  0.9076
AUC:       0.9887

Confusion Matrix:
                 Predicted
              Ham    Spam
Actual Ham    452    0     
       Spam   11     54    

Performance on Test Set - Logistic Regression
Accuracy:  0.9632
Precision: 0.9600
Recall:    0.7385
F1-Score:  0.8348
AUC:       0.9939

Confusion Matrix:
                 Predicted
              Ham    Spam
Actual Ham    450    2     
       Spam   17     48    

Performance on Test Set - SVM
Accuracy:  0.9787
Precision: 0.9821
Recall:    0.8462
F1-Score:  0.9091
AUC:       0.9959

Confusion Matrix:
                 Predicted
              Ham    Spam
Actual Ham    451    1     
       Spam   10     55    

Performance on Test Set - Tuned SVM
Accuracy:  0.9787
Precision: 0.9821
Recall:    0.8462
F1-Score:  0.9091
AUC:       0.9959

Confusion Matrix:
                 Predicted
              Ham    Spam
Act

## 8. Select Best Model

In [16]:
# Create final comparison table
final_comparison = pd.DataFrame({
    'Model': [name for name, _ in test_results],
    'Accuracy': [metrics['accuracy'] for _, metrics in test_results],
    'Precision': [metrics['precision'] for _, metrics in test_results],
    'Recall': [metrics['recall'] for _, metrics in test_results],
    'F1-Score': [metrics['f1_score'] for _, metrics in test_results],
    'AUC': [metrics.get('auc', 0) for _, metrics in test_results]
})

print("\n" + "="*100)
print("FINAL TEST SET COMPARISON")
print("="*100)
print(final_comparison.to_string(index=False))
print("="*100)

# Select best model based on F1-score
best_idx = final_comparison['F1-Score'].idxmax()
best_model_name = final_comparison.loc[best_idx, 'Model']
best_f1 = final_comparison.loc[best_idx, 'F1-Score']

print(f"\n🏆 BEST MODEL: {best_model_name}")
print(f"   F1-Score: {best_f1:.4f}")
print(f"   Accuracy: {final_comparison.loc[best_idx, 'Accuracy']:.4f}")
print(f"   Precision: {final_comparison.loc[best_idx, 'Precision']:.4f}")
print(f"   Recall: {final_comparison.loc[best_idx, 'Recall']:.4f}")

# Model selection mapping
model_map = {
    'Naive Bayes': nb_fitted,
    'Logistic Regression': lr_fitted,
    'SVM': svm_fitted,
    'Tuned LR': best_lr_model
}

selected_model = model_map[best_model_name]
print(f"\nSelected model object: {type(selected_model).__name__}")


FINAL TEST SET COMPARISON
              Model  Accuracy  Precision   Recall  F1-Score      AUC
        Naive Bayes  0.978723   1.000000 0.830769  0.907563 0.988700
Logistic Regression  0.963250   0.960000 0.738462  0.834783 0.993873
                SVM  0.978723   0.982143 0.846154  0.909091 0.995916
          Tuned SVM  0.978723   0.982143 0.846154  0.909091 0.995916

🏆 BEST MODEL: SVM
   F1-Score: 0.9091
   Accuracy: 0.9787
   Precision: 0.9821
   Recall: 0.8462

Selected model object: LinearSVC


## 9. Model Registration and Version Control with MLflow

In [ ]:
### 9.1 Register Models in MLflow Registry
# Register the three benchmark models
model_runs = {
    'Naive Bayes': nb_run_id,
    'Logistic Regression': lr_run_id,
    'SVM': svm_run_id
}

registered_models = {}

for model_name, run_id in model_runs.items():
    try:
        model_uri = f"runs:/{run_id}/{model_name}_model"
        
        # Register model in MLflow registry
        mv = mlflow.register_model(model_uri, f"sms-spam-{model_name.lower().replace(' ', '-')}")
        registered_models[model_name] = mv.name
        
        print(f"✓ {model_name} registered as '{mv.name}'")
        print(f"  Version: {mv.version}")
    except Exception as e:
        print(f"Note: {model_name} may already be registered: {e}")
        registered_models[model_name] = f"sms-spam-{model_name.lower().replace(' ', '-')}"

### 9.2 View Benchmark Models Performance Metrics from MLflow

In [ ]:
# Retrieve metrics from MLflow for each model
print("\n" + "="*100)
print("BENCHMARK MODELS AUCPR METRICS (FROM MLFLOW)")
print("="*100)

mlflow_metrics = {}

for model_name, run_id in model_runs.items():
    run = mlflow.get_run(run_id)
    
    train_aucpr = run.data.metrics.get('train_aucpr', 0)
    val_aucpr = run.data.metrics.get('val_aucpr', 0)
    test_aucpr = run.data.metrics.get('test_aucpr', 0)
    
    mlflow_metrics[model_name] = {
        'train_aucpr': train_aucpr,
        'val_aucpr': val_aucpr,
        'test_aucpr': test_aucpr
    }
    
    print(f"\n{model_name}:")
    print(f"  Run ID: {run_id}")
    print(f"  Train AUCPR: {train_aucpr:.4f}")
    print(f"  Val AUCPR:   {val_aucpr:.4f}")
    print(f"  Test AUCPR:  {test_aucpr:.4f}")

# Create comparison table
comparison_aucpr_df = pd.DataFrame({
    'Model': list(mlflow_metrics.keys()),
    'Train AUCPR': [mlflow_metrics[m]['train_aucpr'] for m in mlflow_metrics.keys()],
    'Val AUCPR': [mlflow_metrics[m]['val_aucpr'] for m in mlflow_metrics.keys()],
    'Test AUCPR': [mlflow_metrics[m]['test_aucpr'] for m in mlflow_metrics.keys()]
})

print("\n" + "="*100)
print("AUCPR COMPARISON TABLE")
print("="*100)
print(comparison_aucpr_df.to_string(index=False))
print("="*100)

### 9.3 Checkout Benchmark Model 1: Naive Bayes

In [ ]:
# Checkout Naive Bayes model from MLflow
print("\n" + "="*100)
print("BENCHMARK MODEL 1: NAIVE BAYES")
print("="*100)

naive_bayes_model_uri = f"runs:{nb_run_id}/Naive Bayes_model"
loaded_nb = mlflow.sklearn.load_model(naive_bayes_model_uri)

nb_run = mlflow.get_run(nb_run_id)
print(f"\nModel: Naive Bayes")
print(f"Run ID: {nb_run_id}")
print(f"Model Type: {nb_run.data.params.get('model_type', 'N/A')}")

print(f"\nMetrics Summary:")
print(f"  Train AUCPR: {nb_run.data.metrics.get('train_aucpr', 0):.4f}")
print(f"  Val AUCPR:   {nb_run.data.metrics.get('val_aucpr', 0):.4f}")
print(f"  Test AUCPR:  {nb_run.data.metrics.get('test_aucpr', 0):.4f}")

print(f"\nValidation Set Metrics:")
print(f"  Accuracy:  {nb_run.data.metrics.get('val_accuracy', 0):.4f}")
print(f"  Precision: {nb_run.data.metrics.get('val_precision', 0):.4f}")
print(f"  Recall:    {nb_run.data.metrics.get('val_recall', 0):.4f}")
print(f"  F1-Score:  {nb_run.data.metrics.get('val_f1_score', 0):.4f}")
print(f"  AUC:       {nb_run.data.metrics.get('val_auc', 0):.4f}")

print(f"✓ Naive Bayes model successfully loaded from MLflow")

### 9.4 Checkout Benchmark Model 2: Logistic Regression

In [ ]:
# Checkout Logistic Regression model from MLflow
print("\n" + "="*100)
print("BENCHMARK MODEL 2: LOGISTIC REGRESSION")
print("="*100)

lr_model_uri = f"runs:{lr_run_id}/Logistic Regression_model"
loaded_lr = mlflow.sklearn.load_model(lr_model_uri)

lr_run = mlflow.get_run(lr_run_id)
print(f"\nModel: Logistic Regression")
print(f"Run ID: {lr_run_id}")
print(f"Model Type: {lr_run.data.params.get('model_type', 'N/A')}")

print(f"\nMetrics Summary:")
print(f"  Train AUCPR: {lr_run.data.metrics.get('train_aucpr', 0):.4f}")
print(f"  Val AUCPR:   {lr_run.data.metrics.get('val_aucpr', 0):.4f}")
print(f"  Test AUCPR:  {lr_run.data.metrics.get('test_aucpr', 0):.4f}")

print(f"\nValidation Set Metrics:")
print(f"  Accuracy:  {lr_run.data.metrics.get('val_accuracy', 0):.4f}")
print(f"  Precision: {lr_run.data.metrics.get('val_precision', 0):.4f}")
print(f"  Recall:    {lr_run.data.metrics.get('val_recall', 0):.4f}")
print(f"  F1-Score:  {lr_run.data.metrics.get('val_f1_score', 0):.4f}")
print(f"  AUC:       {lr_run.data.metrics.get('val_auc', 0):.4f}")

print(f"✓ Logistic Regression model successfully loaded from MLflow")

### 9.5 Checkout Benchmark Model 3: SVM (Support Vector Machine)

In [ ]:
# Checkout SVM model from MLflow
print("\n" + "="*100)
print("BENCHMARK MODEL 3: SVM (SUPPORT VECTOR MACHINE)")
print("="*100)

svm_model_uri = f"runs:{svm_run_id}/SVM_model"
loaded_svm = mlflow.sklearn.load_model(svm_model_uri)

svm_run = mlflow.get_run(svm_run_id)
print(f"\nModel: SVM")
print(f"Run ID: {svm_run_id}")
print(f"Model Type: {svm_run.data.params.get('model_type', 'N/A')}")

print(f"\nMetrics Summary:")
print(f"  Train AUCPR: {svm_run.data.metrics.get('train_aucpr', 0):.4f}")
print(f"  Val AUCPR:   {svm_run.data.metrics.get('val_aucpr', 0):.4f}")
print(f"  Test AUCPR:  {svm_run.data.metrics.get('test_aucpr', 0):.4f}")

print(f"\nValidation Set Metrics:")
print(f"  Accuracy:  {svm_run.data.metrics.get('val_accuracy', 0):.4f}")
print(f"  Precision: {svm_run.data.metrics.get('val_precision', 0):.4f}")
print(f"  Recall:    {svm_run.data.metrics.get('val_recall', 0):.4f}")
print(f"  F1-Score:  {svm_run.data.metrics.get('val_f1_score', 0):.4f}")
print(f"  AUC:       {svm_run.data.metrics.get('val_auc', 0):.4f}")

print(f"✓ SVM model successfully loaded from MLflow")

### 9.6 MLflow Experiments Summary

In [ ]:
# Display MLflow tracking information
print("\n" + "="*100)
print("MLFLOW EXPERIMENT TRACKING SUMMARY")
print("="*100)

experiment = mlflow.get_experiment_by_name("SMS-Spam-Classification")
print(f"\nExperiment: {experiment.name}")
print(f"Experiment ID: {experiment.experiment_id}")
print(f"Artifact Location: {experiment.artifact_location}")

# Get all runs
all_runs = mlflow.search_runs(experiment_ids=[experiment.experiment_id])
print(f"\nTotal Runs Created: {len(all_runs)}")

print(f"\nBenchmark Models Tracked:")
for i, (model_name, run_id) in enumerate(model_runs.items(), 1):
    run = mlflow.get_run(run_id)
    aucpr = run.data.metrics.get('val_aucpr', 0)
    print(f"  {i}. {model_name}")
    print(f"     Run ID: {run_id}")
    print(f"     Validation AUCPR: {aucpr:.4f}")

print(f"\n✓ All models successfully tracked and registered with MLflow!")
print(f"✓ MLflow tracking URI: {mlflow.get_tracking_uri()}")
print(f"✓ Use 'mlflow ui' to view experiment details in the web interface")
print("="*100)